# ML baselines — Drug-disjoint (compound-disjoint) grouped CV

`03_ML_baselines_nestedCV.ipynb` 의 특징 추출·모델·지표를 그대로 쓰고, 분할만
`StratifiedGroupKFold(groups = Drug)` 로 바꿉니다. 같은 약물의 레코드는 모두 같은 파티션에
들어가므로, held-out fold 의 약물은 학습에 한 번도 등장하지 않습니다.

**멀티모달과 짝지은 비교를 하려면 fold 가 동일해야 합니다.** 멀티모달 drug-disjoint 노트북이
저장한 fold 배정 CSV 가 있으면 `FOLD_ASSIGNMENT_CSV` 에 경로를 주세요. 그 파일을 그대로
읽어 씁니다. 없으면 아래 설정값으로 새로 만들고 배정표를 저장하므로, 멀티모달 쪽에서도
같은 파일을 읽게 하면 됩니다.

출력: `ml_drug_grouped_fold_metrics.csv`, `ml_drug_grouped_summary.csv`,
`ml_drug_grouped_oof_predictions.csv`, `drug_grouped_fold_assignments.csv`

## 1. 설정

In [1]:
DATA_CSV      = 'dataset.csv'
ONEHOT_FOLDER = 'onehot_7'
GROUP_COLUMN  = 'Drug'        # 'Variant' 로 바꾸면 diplotype-grouped 가 된다

# 멀티모달 drug-disjoint 실행이 저장한 fold 배정 CSV (없으면 None)
# 열: 최소한 GROUP_COLUMN 과 'fold'. 행 순서가 DATA_CSV 와 같으면 fold 열만 그대로 쓴다.
FOLD_ASSIGNMENT_CSV = None

N_SPLITS     = 5              # 멀티모달 drug-grouped 실행과 동일
RANDOM_STATE = 2              # 멀티모달 drug-grouped 실행의 SEED 와 동일

TARGET_ROWS_SINGLE = 4330
TARGET_COLS        = 7
ECFP_BITS          = 1024
SVD_MAX_COMPONENTS = 500
N_CLASSES          = 3
AVERAGE            = 'macro'  # DL 과 동일
METRICS = ['Accuracy', 'Precision', 'Recall', 'F1', 'AUROC', 'AUPR']
CLASSES = [0, 1, 2]

In [2]:
import os, warnings
import numpy as np
import pandas as pd
from sklearn.preprocessing import LabelEncoder, StandardScaler, label_binarize
from sklearn.decomposition import TruncatedSVD
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.metrics import (accuracy_score, precision_score, recall_score,
                             f1_score, roc_auc_score, average_precision_score,
                             confusion_matrix, classification_report)
from xgboost import XGBClassifier
from rdkit import Chem, RDLogger
from rdkit.Chem import AllChem, DataStructs

RDLogger.DisableLog('rdApp.*')
warnings.filterwarnings('ignore', category=UserWarning)

## 2. 데이터 + 그룹 정의

In [3]:
all_data = pd.read_csv(DATA_CSV)
label_encoder = LabelEncoder()
all_data['Class'] = label_encoder.fit_transform(all_data['Class'])
class_mapping = {i: c for i, c in enumerate(label_encoder.classes_)}
y_full = all_data['Class'].values

def canonical_diplotype(v):
    return '; '.join(sorted(x.strip() for x in str(v).split(';') if x.strip()))

if GROUP_COLUMN == 'Variant':
    groups = all_data['Variant'].map(canonical_diplotype).values
else:
    # 멀티모달 노트북과 동일한 표기: strip 하지 않는다 (그룹 키가 달라지면 fold 가 어긋남)
    groups = all_data[GROUP_COLUMN].astype(str).values

print('label mapping:', class_mapping)
print('샘플 수:', len(all_data), '| 그룹 수 (', GROUP_COLUMN, '):', len(set(groups)))
print(all_data['Class'].value_counts().sort_index().to_string())

label mapping: {0: 'de', 1: 'f', 2: 'nor'}
샘플 수: 622 | 그룹 수 ( Drug ): 59
0    250
1    195
2    177


## 3. 특징 — one-hot 합산 + ECFP (메인 베이스라인과 동일)

In [4]:
onehot_matrices = {}
for file in os.listdir(ONEHOT_FOLDER):
    if not file.endswith('.csv'):
        continue
    key = file.split('.csv')[0].replace('_', '.')
    mat = pd.read_csv(os.path.join(ONEHOT_FOLDER, file), header=None).values
    padded = np.zeros((TARGET_ROWS_SINGLE, TARGET_COLS), dtype=np.float32)
    r = min(mat.shape[0], TARGET_ROWS_SINGLE); c = min(mat.shape[1], TARGET_COLS)
    padded[:r, :c] = mat[:r, :c]
    onehot_matrices[key] = padded
print('one-hot 행렬:', len(onehot_matrices))


def variant_to_sum(variant_str):
    variants = [] if pd.isna(variant_str) else [v.strip() for v in str(variant_str).split(';') if v.strip()]
    mats = [onehot_matrices.get(v, np.zeros((TARGET_ROWS_SINGLE, TARGET_COLS), dtype=np.float32))
            for v in variants[:2]]
    while len(mats) < 2:
        mats.append(np.zeros((TARGET_ROWS_SINGLE, TARGET_COLS), dtype=np.float32))
    return mats[0] + mats[1]


X_seq_full = np.stack([variant_to_sum(v) for v in all_data['Variant']]).astype(np.float64)
X_seq_full = X_seq_full.reshape(len(all_data), -1)
print('seq flat:', X_seq_full.shape)


def smiles_to_ecfp(smiles, n_bits=ECFP_BITS, radius=2):
    mol = Chem.MolFromSmiles(smiles)
    if mol is None:
        return np.zeros((n_bits,), dtype=np.float32)
    fp = AllChem.GetMorganFingerprintAsBitVect(mol, radius=radius, nBits=n_bits)
    arr = np.zeros((n_bits,), dtype=np.float32)
    DataStructs.ConvertToNumpyArray(fp, arr)
    return arr


X_drug_full = np.vstack([smiles_to_ecfp(s) for s in all_data['isomeric']])
print('ECFP:', X_drug_full.shape)

one-hot 행렬: 361
seq flat: (622, 30310)
ECFP: (622, 1024)


## 4. Drug-disjoint fold 배정

`FOLD_ASSIGNMENT_CSV` 가 있으면 그것을 읽고, 없으면 새로 만들어 저장합니다.
어느 경우든 학습 파티션과 held-out 파티션 사이에 그룹이 겹치지 않는지 검증합니다.

In [5]:
n_total = len(all_data)
fold_id = np.full(n_total, -1, dtype=int)

if FOLD_ASSIGNMENT_CSV and os.path.exists(FOLD_ASSIGNMENT_CSV):
    fa = pd.read_csv(FOLD_ASSIGNMENT_CSV)
    if len(fa) == n_total and 'fold' in fa.columns:
        fold_id = fa['fold'].values.astype(int)
        print(f'fold 배정을 {FOLD_ASSIGNMENT_CSV} 에서 행 순서대로 읽었습니다.')
    else:
        key_col = GROUP_COLUMN if GROUP_COLUMN in fa.columns else fa.columns[0]
        mapping = dict(zip(fa[key_col].astype(str), fa['fold'].astype(int)))
        missing = sorted({g for g in groups if g not in mapping})
        assert not missing, f'배정표에 없는 그룹: {missing[:10]}'
        fold_id = np.array([mapping[g] for g in groups], dtype=int)
        print(f'fold 배정을 {FOLD_ASSIGNMENT_CSV} 의 그룹명으로 매칭했습니다.')
else:
    cv = StratifiedGroupKFold(n_splits=N_SPLITS, shuffle=True, random_state=RANDOM_STATE)
    for f, (_tr, _te) in enumerate(cv.split(np.zeros(n_total), y_full, groups)):
        fold_id[_te] = f + 1
    print('fold 배정을 새로 생성했습니다. 멀티모달 쪽에서도 저장된 CSV 를 읽어 쓰세요.')

assert (fold_id > 0).all(), '배정되지 않은 샘플이 있습니다.'

for f in sorted(set(fold_id)):
    te = fold_id == f
    tr = ~te
    overlap = set(groups[tr]) & set(groups[te])
    assert not overlap, f'Fold {f}: 그룹 누수 {sorted(overlap)[:5]}'
    counts = np.bincount(y_full[te], minlength=N_CLASSES)
    print(f'Fold {f}: train={tr.sum()}, test={te.sum()}, '
          f'test groups={len(set(groups[te]))}, overlap=0, class counts={counts.tolist()}')

split_df = all_data[['Variant', 'Drug']].copy()
split_df['group'] = groups
split_df['fold'] = fold_id
split_df.to_csv('drug_grouped_fold_assignments.csv', index=False, encoding='utf-8-sig')
print('saved drug_grouped_fold_assignments.csv')

fold 배정을 새로 생성했습니다. 멀티모달 쪽에서도 저장된 CSV 를 읽어 쓰세요.
Fold 1: train=528, test=94, test groups=12, overlap=0, class counts=[39, 37, 18]
Fold 2: train=503, test=119, test groups=11, overlap=0, class counts=[49, 38, 32]
Fold 3: train=467, test=155, test groups=12, overlap=0, class counts=[52, 51, 52]
Fold 4: train=514, test=108, test groups=11, overlap=0, class counts=[41, 37, 30]
Fold 5: train=476, test=146, test groups=13, overlap=0, class counts=[69, 32, 45]
saved drug_grouped_fold_assignments.csv


## 5. 모델 (메인 베이스라인과 동일한 하이퍼파라미터)

In [6]:
def make_model(name):
    if name == 'Random forest':
        return RandomForestClassifier(
            n_estimators=250, max_depth=30, min_samples_leaf=2,
            min_samples_split=2, n_jobs=-1, random_state=RANDOM_STATE)
    if name == 'SVM':
        return Pipeline([
            ('scaler', StandardScaler()),
            ('svm', SVC(kernel='poly', C=0.22788439100624497, coef0=7.261181986115988,
                        degree=4, gamma=2.6713901787135634e-05,
                        probability=True, random_state=RANDOM_STATE))])
    if name == 'XGBoost':
        return XGBClassifier(
            n_estimators=658, learning_rate=0.048998630084050664, max_depth=5,
            subsample=0.9464704583099741, colsample_bytree=0.6624074561769746,
            objective='multi:softprob', num_class=N_CLASSES, eval_metric='mlogloss',
            tree_method='hist', random_state=RANDOM_STATE)
    raise ValueError(name)


MODEL_NAMES = ['Random forest', 'SVM', 'XGBoost']

## 6. 평가 지표

drug-disjoint 분할에서는 held-out fold 에 세 클래스가 모두 들어있지 않을 수 있습니다.
그 경우 AUROC·AUPR 은 계산이 불가능하므로 NaN 으로 두고, 평균에서 제외합니다.

In [7]:
def metrics_from_pred(y_true, y_pred, y_proba):
    out = {
        'Accuracy':  accuracy_score(y_true, y_pred),
        'Precision': precision_score(y_true, y_pred, average=AVERAGE, zero_division=0),
        'Recall':    recall_score(y_true, y_pred, average=AVERAGE, zero_division=0),
        'F1':        f1_score(y_true, y_pred, average=AVERAGE, zero_division=0),
    }
    present = sorted(set(np.asarray(y_true).tolist()))
    if len(present) < 2:
        out['AUROC'] = np.nan
        out['AUPR'] = np.nan
        return out
    try:
        out['AUROC'] = roc_auc_score(y_true, y_proba, multi_class='ovr',
                                     average=AVERAGE, labels=CLASSES)
    except ValueError:
        out['AUROC'] = np.nan
    aps = []
    for c in CLASSES:
        binary = (np.asarray(y_true) == c).astype(int)
        aps.append(average_precision_score(binary, y_proba[:, c]) if binary.sum() > 0 else np.nan)
    out['AUPR'] = float(np.nanmean(aps))
    return out

## 7. 교차검증

In [8]:
oof = {name: {'probs': np.zeros((n_total, N_CLASSES)), 'pred': np.full(n_total, -1, int)}
       for name in MODEL_NAMES}
rows = []

for f in sorted(set(fold_id)):
    test_idx  = np.where(fold_id == f)[0]
    train_idx = np.where(fold_id != f)[0]

    # 누수 방지: SVD 는 학습 파티션에만 fit
    n_comp = max(2, min(SVD_MAX_COMPONENTS, len(train_idx) - 1, X_seq_full.shape[1] - 1))
    svd = TruncatedSVD(n_components=n_comp, random_state=RANDOM_STATE)
    svd.fit(X_seq_full[train_idx])

    def build(idx):
        return np.hstack([svd.transform(X_seq_full[idx]), X_drug_full[idx]])

    X_tr, X_te = build(train_idx), build(test_idx)
    y_tr, y_te = y_full[train_idx], y_full[test_idx]

    for name in MODEL_NAMES:
        clf = make_model(name)
        clf.fit(X_tr, y_tr)
        y_pred  = clf.predict(X_te)
        y_proba = clf.predict_proba(X_te)
        rows.append({'model': name, 'fold': f, 'n_train': len(train_idx),
                     'n_test': len(test_idx), 'n_test_groups': len(set(groups[test_idx])),
                     **metrics_from_pred(y_te, y_pred, y_proba)})
        oof[name]['probs'][test_idx] = y_proba
        oof[name]['pred'][test_idx]  = y_pred

    print(f'fold {f} done (SVD comp={n_comp}, train={len(train_idx)}, test={len(test_idx)})')

ml_df = pd.DataFrame(rows)
ml_df.to_csv('ml_drug_grouped_fold_metrics.csv', index=False, encoding='utf-8-sig')
print('\nsaved ml_drug_grouped_fold_metrics.csv', ml_df.shape)
ml_df.round(4)

fold 1 done (SVD comp=500, train=528, test=94)
fold 2 done (SVD comp=500, train=503, test=119)
fold 3 done (SVD comp=466, train=467, test=155)
fold 4 done (SVD comp=500, train=514, test=108)
fold 5 done (SVD comp=475, train=476, test=146)

saved ml_drug_grouped_fold_metrics.csv (15, 11)


,model,fold,n_train,n_test,n_test_groups,Accuracy,Precision,Recall,F1,AUROC,AUPR
0,Random forest,1,528,94,12,0.7872,0.7786,0.7859,0.7792,0.9348,0.8900
1,SVM,1,528,94,12,0.7447,0.7296,0.6344,0.6110,0.8594,0.6852
2,XGBoost,1,528,94,12,0.4149,0.3030,0.3347,0.2355,0.8335,0.7053
3,Random forest,2,503,119,11,0.8403,0.8405,0.8432,0.8417,0.9599,0.9339
4,SVM,2,503,119,11,0.6218,0.7068,0.6295,0.5928,0.9253,0.8880
5,XGBoost,2,503,119,11,0.8655,0.8684,0.8672,0.8677,0.9679,0.9464
6,Random forest,3,467,155,12,0.7806,0.8511,0.7809,0.7824,0.9434,0.8903
7,SVM,3,467,155,12,0.8194,0.8456,0.8196,0.8240,0.9481,0.9039
8,XGBoost,3,467,155,12,0.8194,0.8689,0.8191,0.8245,0.9620,0.9318
9,Random forest,4,514,108,11,0.8241,0.8821,0.8129,0.8273,0.9685,0.9474


## 8. 요약 + out-of-fold 예측 저장

In [9]:
summary_rows = []
for name in MODEL_NAMES:
    sub = ml_df[ml_df['model'] == name]
    pooled = metrics_from_pred(y_full, oof[name]['pred'], oof[name]['probs'])
    row = {'model': name}
    print(f'\n=== {name} ===')
    for m in METRICS:
        mean, sd = sub[m].mean(), sub[m].std(ddof=1)
        row[f'{m}_mean'] = mean
        row[f'{m}_sd'] = sd
        row[f'{m}_pooled'] = pooled[m]
        print(f'  {m}: {mean:.4f} ± {sd:.4f}   (pooled OOF {pooled[m]:.4f})')
    summary_rows.append(row)

summary = pd.DataFrame(summary_rows)
summary.to_csv('ml_drug_grouped_summary.csv', index=False, encoding='utf-8-sig')

oof_df = all_data[['Variant', 'Drug']].copy()
oof_df['group'] = groups
oof_df['fold'] = fold_id
oof_df['y_true'] = [class_mapping[c] for c in y_full]
for name in MODEL_NAMES:
    tag = name.replace(' ', '_')
    oof_df[f'{tag}_pred'] = [class_mapping[c] for c in oof[name]['pred']]
    for i in CLASSES:
        oof_df[f'{tag}_prob_{class_mapping[i]}'] = oof[name]['probs'][:, i]
oof_df.to_csv('ml_drug_grouped_oof_predictions.csv', index=False, encoding='utf-8-sig')

print('\nsaved ml_drug_grouped_summary.csv, ml_drug_grouped_oof_predictions.csv')
summary.round(4)


=== Random forest ===
  Accuracy: 0.8122 ± 0.0266   (pooled OOF 0.8119)
  Precision: 0.8378 ± 0.0376   (pooled OOF 0.8313)
  Recall: 0.8140 ± 0.0309   (pooled OOF 0.8061)
  F1: 0.8141 ± 0.0309   (pooled OOF 0.8153)
  AUROC: 0.9497 ± 0.0140   (pooled OOF 0.9422)
  AUPR: 0.9126 ± 0.0265   (pooled OOF 0.8967)

=== SVM ===
  Accuracy: 0.7347 ± 0.0731   (pooled OOF 0.7379)
  Precision: 0.7735 ± 0.0645   (pooled OOF 0.7561)
  Recall: 0.7190 ± 0.0842   (pooled OOF 0.7364)
  F1: 0.7009 ± 0.0973   (pooled OOF 0.7437)
  AUROC: 0.9129 ± 0.0358   (pooled OOF 0.8950)
  AUPR: 0.8341 ± 0.0930   (pooled OOF 0.7974)

=== XGBoost ===
  Accuracy: 0.7045 ± 0.1755   (pooled OOF 0.7235)
  Precision: 0.7072 ± 0.2339   (pooled OOF 0.7524)
  Recall: 0.6979 ± 0.2139   (pooled OOF 0.7180)
  F1: 0.6686 ± 0.2525   (pooled OOF 0.7293)
  AUROC: 0.9243 ± 0.0556   (pooled OOF 0.9066)
  AUPR: 0.8674 ± 0.0992   (pooled OOF 0.8466)

saved ml_drug_grouped_summary.csv, ml_drug_grouped_oof_predictions.csv


,model,Accuracy_mean,Accuracy_sd,Accuracy_pooled,Precision_mean,Precision_sd,Precision_pooled,Recall_mean,Recall_sd,Recall_pooled,F1_mean,F1_sd,F1_pooled,AUROC_mean,AUROC_sd,AUROC_pooled,AUPR_mean,AUPR_sd,AUPR_pooled
0,Random forest,0.8122,0.0266,0.8119,0.8378,0.0376,0.8313,0.8140,0.0309,0.8061,0.8141,0.0309,0.8153,0.9497,0.0140,0.9422,0.9126,0.0265,0.8967
1,SVM,0.7347,0.0731,0.7379,0.7735,0.0645,0.7561,0.7190,0.0842,0.7364,0.7009,0.0973,0.7437,0.9129,0.0358,0.8950,0.8341,0.0930,0.7974
2,XGBoost,0.7045,0.1755,0.7235,0.7072,0.2339,0.7524,0.6979,0.2139,0.7180,0.6686,0.2525,0.7293,0.9243,0.0556,0.9066,0.8674,0.0992,0.8466
